# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print basic info
print(f"Dataset name: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we list all record sets available in the dataset, along with their `@id`s and included fields (and associated columns) with their `@id`s.

In [ ]:
# List all record sets with their @id and fields
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}")
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}, name: {getattr(rs, 'name', None)}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    Field @id: {field.id}, name: {getattr(field, 'name', None)} (dataType: {getattr(field, 'data_type', None)})")
        if hasattr(field, 'columns'):
            for col in field.columns:
                print(f"      Column @id: {col.id}, name: {getattr(col, 'name', None)}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

Below, we extract all record sets into DataFrames and print the columns for one of the record sets.

In [ ]:
# Extract data from each record set
record_sets = dataset.record_sets
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]

for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    dataframes[rs.id] = pd.DataFrame(records)

# For demonstration, choose the first record set
chosen_record_set_id = record_set_ids[0] if record_set_ids else None
if chosen_record_set_id and chosen_record_set_id in dataframes:
    print(f"Columns in record set {@id}: {chosen_record_set_id}")
    print(dataframes[chosen_record_set_id].columns.tolist())
    display(dataframes[chosen_record_set_id].head())
else:
    print("No record sets with data available.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

_NOTE_: All entity references use their `@id` in variables. Make sure to set `numeric_field_id` and `group_field_id` to match the `@id`s as explored above.

In [ ]:
# Example values for demonstration (you should set these after inspecting the overview, replacing as needed):
record_set_id = chosen_record_set_id  # The @id of the RecordSet to analyze (from overview)

# SHOW the first 5 column ids
print(f"First 5 columns: {dataframes[record_set_id].columns[:5]}")

# Example: Suppose the 'cr:coverage' and 'cr:description' fields exist
# Replace 'cr:coverage'/'cr:description' with actual @id from the overview above as appropriate
numeric_field_id = None
group_field_id = None

# Try to automatically pick a numeric field for demonstration (float/integer columns)
df = dataframes[record_set_id]
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break
# Try to pick a group-able (string) field
for col in df.columns:
    if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id:
        group_field_id = col
        break

print(f"Numeric field selected (by @id): {numeric_field_id}")
print(f"Group-by field selected (by @id): {group_field_id}")

if numeric_field_id:
    # Remove NaNs
    valid_numeric = df[numeric_field_id].dropna()
    threshold = valid_numeric.mean() if not valid_numeric.empty else 0
    print(f"Applying threshold > {threshold:.2f} to field '{numeric_field_id}'")
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of '{numeric_field_id}' by '{group_field_id}':")
        display(grouped_df)
else:
    print("No numeric field identified for EDA. Please inspect the DataFrame and set 'numeric_field_id'.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Update the field `@id`s to match actual fields.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if numeric_field_id:
    plt.figure(figsize=(8,5))
    df[numeric_field_id].hist(bins=30)
    plt.title(f"Distribution of field: {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id:
        filtered_df.boxplot(column=numeric_field_id, by=group_field_id, grid=False, vert=True, figsize=(10,6))
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.suptitle("")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("Not enough numeric data for visualization!")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

**Summary:**
- We loaded the Mass Spectrometry Analysis of Extracellular Vesicles dataset using `mlcroissant` by schema URL.
- All entities (record sets, fields, columns) were referenced and inspected via their `@id`, ensuring stable usage.
- We explored the dataset's structure, loaded records for analysis, and performed simple EDA and visualizations using numeric and group-by fields.

**Next steps:**
- Identify key domain fields using their `@id`s for deeper modeling.
- Clean, augment, or transform fields further as needed for ML tasks.